In [127]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import (
    TimeSeriesSplit,
    RandomizedSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

import joblib
import warnings

warnings.filterwarnings("ignore")

In [128]:
X_train = joblib.load("Demand_X_train.pkl")
X_test = joblib.load("Demand_X_test.pkl")

y_train = joblib.load("Demand_y_train.pkl")
y_test = joblib.load("Demand_y_test.pkl")

In [129]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (100, 18)
X_test  : (26, 18)
y_train : (100,)
y_test  : (26,)


In [130]:
# Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=5)

# Random Forest Model
rf = RandomForestRegressor(random_state=42)

# Hyperparameter Search Space
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700],
    'max_depth': [5, 8, 10, 15, 20, None],
    'min_samples_split': [2, 3, 5, 8],
    'min_samples_leaf': [1, 2, 3, 4],
    'max_features': ['sqrt', 'log2', 0.7, 1.0],
    'bootstrap': [True, False]
}

# Random Search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=tscv,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

# Train Model
random_search.fit(X_train, y_train)

# Best Model
rf_model = random_search.best_estimator_

print("Best Parameters:")
print(random_search.best_params_)

Best Parameters:
{'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.7, 'max_depth': 8, 'bootstrap': False}


In [131]:
rf_model.fit(X_train, y_train)

RandomForestRegressor(bootstrap=False, max_depth=8, max_features=0.7,
                      min_samples_leaf=2, min_samples_split=5, n_estimators=500,
                      random_state=42)

In [132]:
y_pred = rf_model.predict(X_test)

In [133]:
print(y_pred)

[ 9472.82905     8931.64386667  8999.08693714  9449.25901429
 11204.20883333 11218.3655     10945.38466667 10939.80466667
 10942.05433333 10701.07233333 10807.7935     10353.98031667
 10092.72685     8902.2286      9292.11006429  9348.13803095
 11147.7325     11256.94633333 11059.17783333 10903.71466667
 10988.7155     10867.6015     10829.73483333 10382.62285556
 10148.74010556 10292.7137    ]


In [134]:
prediction_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})
prediction_df.head(10)

,Actual,Predicted
0,8759.0,9472.829050
1,9356.0,8931.643867
2,9658.0,8999.086937
3,10404.0,9449.259014
4,10452.0,11204.208833
5,12649.0,11218.365500
6,11838.0,10945.384667
7,10784.0,10939.804667
8,11192.0,10942.054333
9,11135.0,10701.072333


In [135]:
# Predictions
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Training Metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Testing Metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("="*50)
print("Training Performance")
print("="*50)
print(f"MAE  : {train_mae:.2f}")
print(f"RMSE : {train_rmse:.2f}")
print(f"MAPE : {train_mape*100:.2f}%")
print(f"R²   : {train_r2:.4f}")

print("\n")

print("="*50)
print("Testing Performance")
print("="*50)
print(f"MAE  : {test_mae:.2f}")
print(f"RMSE : {test_rmse:.2f}")
print(f"MAPE : {test_mape*100:.2f}%")
print(f"R²   : {test_r2:.4f}")

Training Performance
MAE  : 58.08
RMSE : 115.09
MAPE : 0.68%
R²   : 0.9849


Testing Performance
MAE  : 554.23
RMSE : 656.99
MAPE : 5.12%
R²   : 0.5535


In [136]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

             Feature  Importance
15      Demand_Lag_1    0.303264
11       Temperature    0.226189
3               Coal    0.162314
0           Humidity    0.072223
7              Solar    0.062179
14         Month_cos    0.039097
2   Solar_Irradiance    0.029536
10         Bio Power    0.019780
8               Wind    0.018244
4          Oil & Gas    0.015117
1           Rainfall    0.012071
12              Year    0.010399
5            Nuclear    0.007299
16      Demand_Lag_2    0.006486
13         Month_sin    0.004805
17      Demand_Lag_3    0.003990
9        Small-Hydro    0.003904
6              Hydro    0.003105


In [137]:
print(X_train.describe())

         Humidity    Rainfall  Solar_Irradiance           Coal    Oil & Gas  \
count  100.000000  100.000000        100.000000     100.000000   100.000000   
mean    74.160400   95.606726        152.503900   86180.122500  3656.020600   
std      7.578832   75.634605         23.807573   10784.414885   907.839618   
min     56.410000    0.786286         79.840000   61916.560000  1590.280000   
25%     69.722500   37.067500        139.827500   78799.237500  3219.535000   
50%     74.520000   82.542714        155.470000   83920.405000  3896.360000   
75%     79.937500  139.723929        167.277500   91654.375000  4239.350000   
max     89.420000  399.657143        204.470000  113958.880000  5600.020000   

           Nuclear         Hydro         Solar          Wind  Small-Hydro  \
count   100.000000    100.000000    100.000000    100.000000   100.000000   
mean   3511.284400  11916.543600   4377.384400   5116.310900   790.041400   
std     509.331155   4541.326942   2860.758106   2997.468

In [138]:
print(X_test.describe())

        Humidity    Rainfall  Solar_Irradiance           Coal    Oil & Gas  \
count  26.000000   26.000000         26.000000      26.000000    26.000000   
mean   76.338077  107.520901        184.847123  108350.381154  2442.121154   
std     7.761415   76.711440         60.849792    7530.591564   915.719953   
min    56.850000    4.588857         98.660000   96119.810000  1433.560000   
25%    72.005000   41.301643        128.440000  102522.512500  1721.860000   
50%    76.485000   91.552000        186.560000  107762.480000  2385.335000   
75%    82.562500  163.227786        228.634221  114285.877500  2634.130000   
max    86.440000  257.944286        303.305385  122574.210000  5163.850000   

           Nuclear         Hydro         Solar          Wind  Small-Hydro  \
count    26.000000     26.000000     26.000000     26.000000    26.000000   
mean   4508.493077  12415.973077  12105.908077   7469.538077   946.390000   
std     451.766990   5859.484886   2085.197728   3949.394233   410